In [1]:
import pandas as pd

import re
import ast
import numpy as np
import pandas as pd
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from typing import Dict, List, Optional, Tuple

C:\Users\shrep\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ─────────────────────────────────────────
# 1. LOAD
# ─────────────────────────────────────────
df = pd.read_csv("nyctastematch-restaurant-recommender/data/processed/restaurants_CA.csv", index_col=0)
print(f"Raw rows: {len(df)}")

# ─────────────────────────────────────────
# 2. FILTER TO FOOD/RESTAURANT BUSINESSES
# ─────────────────────────────────────────
# All category tags from the dataset that relate to food/drink
FOOD_CATEGORIES = {
    # Core
    "Restaurants", "Food",
    # Cuisine types
    "American (New)", "American (Traditional)", "Arabic", "Argentine",
    "Asian Fusion", "Australian", "Barbeque", "Basque", "Belgian",
    "Brazilian", "British", "Buffets", "Burgers", "Cafeteria",
    "Cajun/Creole", "Cantonese", "Caribbean", "Cheesesteaks",
    "Chicken Shop", "Chicken Wings", "Chinese", "Comfort Food",
    "Cuban", "Dim Sum", "Diners", "Ethiopian", "Falafel",
    "Fast Food", "Fish & Chips", "Fondue", "French", "German",
    "Greek", "Hawaiian", "Himalayan/Nepalese", "Hot Dogs", "Hot Pot",
    "Indian", "Indonesian", "Irish", "Italian", "Japanese",
    "Kebab", "Korean", "Latin American", "Lebanese",
    "Mediterranean", "Mexican", "Middle Eastern", "Modern European",
    "Mongolian", "Moroccan", "New Mexican Cuisine", "Noodles",
    "Pakistani", "Pan Asian", "Persian/Iranian", "Peruvian",
    "Pizza", "Poke", "Ramen", "Salad", "Salvadoran", "Sandwiches",
    "Scandinavian", "Seafood", "Smokehouse", "Soul Food", "Soup",
    "Southern", "Spanish", "Steakhouses", "Sushi Bars",
    "Szechuan", "Tacos", "Taiwanese", "Tapas/Small Plates",
    "Tex-Mex", "Thai", "Turkish", "Tuscan", "Vietnamese", "Wraps",
    # Drink & casual
    "Bars", "Beer Bar", "Beer Gardens", "Breweries", "Brewpubs",
    "Cideries", "Cocktail Bars", "Coffee & Tea", "Coffeeshops",
    "Distilleries", "Dive Bars", "Food Court", "Food Stands",
    "Food Trucks", "Gastropubs", "Gay Bars", "Hookah Bars",
    "Irish Pub", "Jazz & Blues", "Juice Bars & Smoothies",
    "Karaoke", "Lounges", "Piano Bars", "Pop-Up Restaurants",
    "Pubs", "Speakeasies", "Sports Bars", "Tapas Bars",
    "Tea Rooms", "Tiki Bars", "Whiskey Bars", "Wine Bars",
    # Food subtypes
    "Acai Bowls", "Bagels", "Bakeries", "Bubble Tea", "Butcher",
    "Candy Stores", "Cheese Shops", "Chocolatiers & Shops",
    "Creperies", "Cupcakes", "Custom Cakes", "Delicatessen",
    "Delis", "Desserts", "Do-It-Yourself Food", "Donuts",
    "Ethnic Food", "Ethnic Grocery", "Food Delivery Services",
    "Fruits & Veggies", "Gelato", "Gluten-Free", "Halal",
    "Health Markets", "Hong Kong Style Cafe", "Ice Cream & Frozen Yogurt",
    "International Grocery", "Live/Raw Food", "Macarons",
    "Meat Shops", "Organic Stores", "Pasta Shops",
    "Patisserie/Cake Shop", "Personal Chefs", "Pretzels",
    "Shaved Ice", "Specialty Food", "Street Vendors",
    "Vegan", "Vegetarian", "Cafes", "Caterers",
    "Breakfast & Brunch", "Grocery",
    # Ambiguous but food-related
    "Themed Cafes", "Internet Cafes",
}

def has_food_category(cat_str):
    if pd.isna(cat_str):
        return False
    cats = {c.strip() for c in cat_str.split(",")}
    return bool(cats & FOOD_CATEGORIES)

df_food = df[df["categories"].apply(has_food_category)].copy()
print(f"After food filter: {len(df_food)}")

catering_only = df["categories"].str.contains("Caterers", na=False) & \
            ~df["categories"].str.contains("Restaurants", na=False)
df = df[~catering_only].reset_index(drop=True)
print(f"After catering filter: {len(df)}")

# ─────────────────────────────────────────
# 3. OPTIONAL: KEEP ONLY OPEN BUSINESSES
# ─────────────────────────────────────────
df_food = df_food[df_food["is_open"] == 1].copy()
print(f"After open filter: {len(df_food)}")

# ─────────────────────────────────────────
# 4. CLEAN COLUMNS
# ─────────────────────────────────────────
# Fix postal_code float → string
df_food["postal_code"] = df_food["postal_code"].apply(
    lambda x: str(int(x)) if pd.notna(x) else ""
)

# Reset index so row index = ChromaDB document ID
df_food = df_food.reset_index(drop=True)
print(f"\nFinal restaurant count: {len(df_food)}")
print(f"Cities covered: {df_food['city'].nunique()}")
print(f"\nTop 10 cities:\n{df_food['city'].value_counts().head(10)}")
print(f"\nStar distribution:\n{df_food['stars'].value_counts().sort_index()}")

# ─────────────────────────────────────────
# 5. PARSE ATTRIBUTES & EXTRACT PRICE
# ─────────────────────────────────────────
def parse_dict_field(val):
    if isinstance(val, dict):
        return val
    try:
        return ast.literal_eval(str(val))
    except:
        return {}

def extract_price(attrs_str):
    attrs = parse_dict_field(attrs_str)
    val = str(attrs.get("RestaurantsPriceRange2", "")).strip("'\" ")
    try:
        return int(val)
    except:
        return None

df_food["price"] = df_food["attributes"].apply(extract_price)
print(f"\nPrice distribution:\n{df_food['price'].value_counts(dropna=False).sort_index()}")

# ─────────────────────────────────────────
# 6. SAVE CLEANED FILE
# ─────────────────────────────────────────
df_food.to_csv("CA_Restaurants_cleaned.csv", index=False)
print("\nSaved → CA_Restaurants_cleaned.csv")

Raw rows: 5203
After food filter: 1662
After catering filter: 5158
After open filter: 1036

Final restaurant count: 1036
Cities covered: 10

Top 10 cities:
city
Santa Barbara     713
Goleta            172
Carpinteria        84
Isla Vista         29
Montecito          24
Summerland         10
Port Hueneme        1
Santa  Barbara      1
Truckee             1
Santa Maria         1
Name: count, dtype: int64

Star distribution:
stars
1.5     12
2.0     29
2.5     65
3.0     99
3.5    144
4.0    306
4.5    259
5.0    122
Name: count, dtype: int64

Price distribution:
price
1.0    267
2.0    513
3.0     52
4.0     16
NaN    188
Name: count, dtype: int64

Saved → CA_Restaurants_cleaned.csv


In [3]:
# ─────────────────────────────────────────
# 1. QUERY PARSER
# ─────────────────────────────────────────
class QueryParser:
    """
    Parse natural language queries to extract filters and preferences.
    """

    def __init__(self):
        self.price_keywords = {
            'cheap': [1, 2],
            'affordable': [1, 2],
            'budget': [1, 2],
            'budget-friendly': [1, 2],
            'inexpensive': [1, 2],
            'moderate': [2, 3],
            'mid-range': [2, 3],
            'expensive': [3, 4],
            'upscale': [3, 4],
            'fancy': [3, 4],
            'fine dining': [3, 4],
            'luxury': [3, 4],
            'high-end': [3, 4],
            'splurge': [3, 4],
        }

        self.rating_keywords = {
            'excellent': 4.5,
            'amazing': 4.5,
            'outstanding': 4.5,
            'great': 4.0,
            'good': 3.5,
            'highly rated': 4.0,
            'top rated': 4.5,
            'best': 4.5,
            'decent': 3.0,
            'okay': 2.5,
            'popular': 3.5,
            'well-reviewed': 4.0,
        }

        # Cuisine keywords
        self.cuisine_keywords = [
            # Cuisines
            'italian', 'chinese', 'japanese', 'mexican', 'indian',
            'thai', 'french', 'american', 'mediterranean', 'greek',
            'korean', 'vietnamese', 'spanish', 'middle eastern',
            'caribbean', 'ethiopian', 'brazilian', 'peruvian',
            'latin', 'salvadoran', 'turkish', 'moroccan',
            'himalayan', 'nepalese', 'indonesian', 'taiwanese',
            'cantonese', 'szechuan', 'pan asian', 'asian fusion',
            # Food types
            'sushi', 'pizza', 'burger', 'seafood', 'steakhouse',
            'ramen', 'noodles', 'hot pot', 'hotpot', 'tacos',
            'sandwiches', 'salad', 'soup', 'tapas', 'dim sum',
            'barbeque', 'bbq', 'wings', 'chicken', 'poke',
            'cheesesteaks', 'falafel', 'kebab', 'wraps',
            # Dietary
            'vegetarian', 'vegan', 'gluten-free', 'halal',
            # Meal / venue type
            'bakery', 'cafe', 'coffee', 'diner', 'brunch', 'breakfast',
            'dessert', 'ice cream', 'bubble tea', 'juice',
            'sports bar', 'cocktails', 'wine bar', 'beer', 'brewpub',
            'gastropub', 'bar', 'lounge',
            # CA-specific from Yelp categories
            'soul food', 'comfort food', 'buffet', 'food truck',
            'patisserie', 'creperie', 'donuts', 'bagels',
        ]

        # CA cities for location filtering
        self.city_keywords = [
            'santa barbara', 'goleta', 'carpinteria',
            'isla vista', 'montecito', 'summerland',
        ]

    def parse_query(self, query: str) -> Dict:
        query_lower = query.lower()

        filters = {
            'price_filter': None,
            'min_rating': None,
            'cuisine_filter': [],
            'city_filter': None,
            'cleaned_query': query,
        }

        filters['price_filter']   = self._extract_price(query_lower)
        filters['min_rating']     = self._extract_rating(query_lower)
        filters['cuisine_filter'] = self._extract_cuisines(query_lower)
        filters['city_filter']    = self._extract_city(query_lower)
        filters['cleaned_query']  = self._clean_query(query_lower)

        return filters

    def _extract_price(self, query: str) -> Optional[List[int]]:
        dollar_match = re.search(r'\$+', query)
        if dollar_match:
            dollar_count = len(dollar_match.group())
            return list(range(1, min(dollar_count + 1, 5)))

        for keyword, price_range in self.price_keywords.items():
            if keyword in query:
                return price_range

        under_match = re.search(
            r'(?:under|less than|below|max|maximum)\s*\$?(\d+)', query
        )
        if under_match:
            max_price = int(under_match.group(1))
            if max_price <= 15:   return [1]
            elif max_price <= 30: return [1, 2]
            elif max_price <= 60: return [1, 2, 3]
            else:                 return [1, 2, 3, 4]

        return None

    def _extract_rating(self, query: str) -> Optional[float]:
        star_match = re.search(r'(\d+(?:\.\d+)?)\+?\s*(?:star|rating)', query)
        if star_match:
            return float(star_match.group(1))

        for keyword, min_rating in self.rating_keywords.items():
            if keyword in query:
                return min_rating

        above_match = re.search(
            r'(?:above|over|at least|minimum)\s*(\d+(?:\.\d+)?)', query
        )
        if above_match:
            return float(above_match.group(1))

        return None

    def _extract_cuisines(self, query: str) -> List[str]:
        found = []
        for cuisine in self.cuisine_keywords:
            pattern = r'\b' + re.escape(cuisine).replace(r'\ ', r'[\s_-]') + r'\b'
            if re.search(pattern, query):
                found.append(cuisine)
        return found

    def _extract_city(self, query: str) -> Optional[str]:
        """New: extract city name for location-aware filtering."""
        for city in self.city_keywords:
            if city in query:
                return city.title()   # e.g. "santa barbara" → "Santa Barbara"
        return None

    def _clean_query(self, query: str) -> str:
        original_query = query
        food_words = set(self.cuisine_keywords)

        for keyword in self.rating_keywords:
            if keyword.lower() not in food_words:
                query = re.sub(
                    r'\b' + re.escape(keyword) + r'\b', '', query,
                    flags=re.IGNORECASE
                )

        query = re.sub(
            r'(?:under|less than|below|max|maximum|above|over|at least|minimum)\s*\$?\d+',
            '', query
        )
        query = re.sub(r'\d+\+?\s*(?:star|rating)s?', '', query)
        query = re.sub(r'\$+', '', query)

        # Strip city names — they're used as filters, not semantic signal
        for city in self.city_keywords:
            query = re.sub(r'\b' + re.escape(city) + r'\b', '', query,
                           flags=re.IGNORECASE)

        query = ' '.join(query.split()).strip()
        return query if len(query) > 3 else original_query.strip()

    def print_parsed_query(self, filters: Dict):
        print("\n" + "=" * 60)
        print("PARSED QUERY")
        print("=" * 60)
        if filters['price_filter']:
            price_labels = {1: "$", 2: "$$", 3: "$$$", 4: "$$$$"}
            prices = [price_labels[p] for p in filters['price_filter']]
            print(f"💰 Price Range: {', '.join(prices)}")
        if filters['min_rating']:
            print(f"⭐ Minimum Rating: {filters['min_rating']}/5")
        if filters['cuisine_filter']:
            print(f"🍽️  Cuisines: {', '.join(filters['cuisine_filter'])}")
        if filters['city_filter']:
            print(f"📍 City: {filters['city_filter']}")
        print(f"🔍 Semantic Query: '{filters['cleaned_query']}'")
        print("=" * 60 + "\n")


In [4]:
 # ─────────────────────────────────────────
# 2. DESCRIPTION BUILDER
# ─────────────────────────────────────────
def parse_dict_field(val):
    if val is None:
        return {}
    if isinstance(val, dict):
        return val
    try:
        result = ast.literal_eval(str(val))
        return result if isinstance(result, dict) else {}
    except:
        return {}


def create_restaurant_description(row):
    """
    Build rich text description for embedding from CA/Yelp data.
    Mirrors NYC version structure; exploits extra Yelp attribute fields.
    """
    parts = []

    # Basic info
    parts.append(f"Restaurant name: {row['name']}")

    if pd.notna(row.get('city')):
        parts.append(f"Located in {row['city']}, California")

    # Categories → cuisine (replaces NYC 'cuisine' list column)
    if pd.notna(row.get('categories')):
        cats = [
            c.strip().lower()
             .replace('&', 'and')
             .replace('/', ' ')
            for c in str(row['categories']).split(',')
            if c.strip().lower() not in ('food', 'restaurants')
        ]
        if cats:
            parts.append(f"Cuisine types: {', '.join(cats)}")

    # Price
    price_labels = {1: "Budget-friendly", 2: "Moderate",
                    3: "Upscale", 4: "Fine dining"}
    price_val = row.get('price')
    if pd.notna(price_val) and int(price_val) in price_labels:
        parts.append(f"Price range: {price_labels[int(price_val)]}")

    # Rating (Yelp uses 'stars' not 'rating')
    try:
        if pd.notna(row.get('stars')):
            parts.append(f"Rating: {float(row['stars'])}/5 stars")
    except (ValueError, TypeError):
        pass

    # Popularity
    if pd.notna(row.get('review_count')):
        rc = int(row['review_count'])
        if rc > 500:
            parts.append("Very popular restaurant")
        elif rc > 100:
            parts.append("Popular restaurant")

    # Address
    if pd.notna(row.get('address')):
        parts.append(f"Location: {row['address']}")

    # ── Yelp-specific attribute fields (not in NYC data) ──────────
    attrs = parse_dict_field(row.get('attributes', {}))

    ambience = parse_dict_field(attrs.get('Ambience', {}))
    ambience_tags = [k for k, v in ambience.items()
                     if str(v).lower() == 'true']
    if ambience_tags:
        parts.append(f"Ambience: {', '.join(ambience_tags)}")

    noise = str(attrs.get('NoiseLevel', '')).strip("u'\" ")
    if noise and noise not in ('None', ''):
        parts.append(f"Noise level: {noise}")

    good_for = parse_dict_field(attrs.get('GoodForMeal', {}))
    meal_tags = [k for k, v in good_for.items()
                 if str(v).lower() == 'true']
    if meal_tags:
        parts.append(f"Good for: {', '.join(meal_tags)}")

    amenity_map = {
        'OutdoorSeating':          'outdoor seating',
        'RestaurantsDelivery':     'delivery available',
        'RestaurantsTakeOut':      'takeout available',
        'RestaurantsReservations': 'accepts reservations',
        'GoodForKids':             'good for kids',
        'DogsAllowed':             'dog friendly',
        'HappyHour':               'happy hour',
        'HasTV':                   'has TV',
        'WheelchairAccessible':    'wheelchair accessible',
    }
    amenities = [label for key, label in amenity_map.items()
                 if str(attrs.get(key, '')).strip("u'\" ").lower() == 'true']
    if amenities:
        parts.append(f"Features: {', '.join(amenities)}")

    alcohol = str(attrs.get('Alcohol', '')).strip("u'\" ")
    if alcohol and alcohol not in ('none', 'None', ''):
        parts.append(f"Alcohol: {alcohol}")

    wifi = str(attrs.get('WiFi', '')).strip("u'\" ")
    if wifi and wifi not in ('no', 'None', ''):
        parts.append("Has WiFi")

    # Late-night signal from hours
    hours = parse_dict_field(row.get('hours', {}))
    for span in hours.values():
        try:
            close_hr = int(str(span).split('-')[-1].split(':')[0])
            if close_hr >= 21 or close_hr == 0:
                parts.append("Open late")
                break
        except:
            pass

    return '. '.join(parts)

In [5]:
# ─────────────────────────────────────────
# 3. GENERATE EMBEDDINGS
# ─────────────────────────────────────────
def generate_embeddings(df):
    df['description'] = df.apply(create_restaurant_description, axis=1)
    print("\nSample description:")
    print(df['description'].iloc[0])

    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print("\nGenerating embeddings...")
    embeddings = embedding_model.encode(
        df['description'].tolist(),
        show_progress_bar=True,
        convert_to_numpy=True
    )
    print(f"Embeddings shape: {embeddings.shape}")
    np.save('ca_restaurant_embeddings.npy', embeddings)
    print("Saved → ca_restaurant_embeddings.npy")
    return embedding_model, embeddings

In [6]:
# ─────────────────────────────────────────
# 4. CHROMADB COLLECTION
# ─────────────────────────────────────────
def recreate_collection_with_cosine(df, embeddings):
    print("Recreating ChromaDB collection...")

    client = chromadb.PersistentClient(path="./ca_chroma_db")

    try:
        client.delete_collection("ca_restaurants")
        print("Deleted old collection")
    except:
        print("No existing collection to delete")

    collection = client.create_collection(
        name="ca_restaurants",
        metadata={"hnsw:space": "cosine"}
    )
    print("✓ Created collection with cosine distance")

    documents  = df['description'].tolist()
    ids        = [str(i) for i in range(len(df))]

    metadatas = []
    for _, row in df.iterrows():
        # Safely coerce every field — ChromaDB rejects None/NaN in metadata
        try:
            price = int(float(row['price'])) if pd.notna(row.get('price')) else 0
        except (ValueError, TypeError):
            price = 0
        try:
            rating = float(row['stars']) if pd.notna(row.get('stars')) else 0.0
        except (ValueError, TypeError):
            rating = 0.0

        metadatas.append({
            'name':    str(row['name'])    if pd.notna(row.get('name'))    else 'Unknown',
            'price':   price,
            'rating':  rating,
            'address': str(row['address']) if pd.notna(row.get('address')) else '',
            'city':    str(row['city'])    if pd.notna(row.get('city'))    else '',
        })

    batch_size = 500
    for i in range(0, len(documents), batch_size):
        end = min(i + batch_size, len(documents))
        collection.add(
            documents  = documents[i:end],
            embeddings = embeddings[i:end].tolist(),
            ids        = ids[i:end],
            metadatas  = metadatas[i:end],
        )
        print(f"  Batch {i//batch_size + 1}/{(len(documents)-1)//batch_size + 1} added")

    print(f"\n✅ Collection ready: {len(documents)} restaurants")
    return client, collection

    

# ─────────────────────────────────────────
# 5. SEARCH
# ─────────────────────────────────────────
def search_restaurants(query, collection, embedding_model, df, top_k=5, verbose=True):
    parser  = QueryParser()
    filters = parser.parse_query(query)

    if verbose:
        parser.print_parsed_query(filters)

    search_query    = filters['cleaned_query'] or query
    query_embedding = embedding_model.encode(search_query).tolist()

    # Build ChromaDB where clause
    conditions = []
    if filters['price_filter']:
        conditions.append({"price": {"$in": filters['price_filter']}})
    if filters['min_rating']:
        conditions.append({"rating": {"$gte": filters['min_rating']}})
    if filters['city_filter']:
        conditions.append({"city": {"$eq": filters['city_filter']}})

    if len(conditions) > 1:
        where_clause = {"$and": conditions}
    elif len(conditions) == 1:
        where_clause = conditions[0]
    else:
        where_clause = None

    n_results = top_k * 4 if filters['cuisine_filter'] else top_k

    query_kwargs = dict(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["distances", "metadatas", "documents"]
    )
    if where_clause:
        query_kwargs["where"] = where_clause

    results = collection.query(**query_kwargs)

    # Post-retrieval cuisine filter
    candidates = list(zip(
        results['ids'][0],
        results['distances'][0],
        results['metadatas'][0],
        results['documents'][0],
    ))

    if filters['cuisine_filter']:
        def matches_cuisine(doc):
            return any(c in doc.lower() for c in filters['cuisine_filter'])
        filtered = [c for c in candidates if matches_cuisine(c[3])]
        candidates = filtered if filtered else candidates

    # Pad to top_k if cuisine filter left too few
    if len(candidates) < top_k:
        all_candidates = list(zip(
            results['ids'][0],
            results['distances'][0],
            results['metadatas'][0],
            results['documents'][0],
        ))
        existing_ids = {c[0] for c in candidates}
        extras = [c for c in all_candidates if c[0] not in existing_ids]
        candidates = (candidates + extras)[:top_k]

    candidates = candidates[:top_k]

    # Build output — pull cuisine + review_count from df by row index
    output = []
    for rid, dist, meta, doc in candidates:
        try:
            row       = df.iloc[int(rid)]
            cuisine   = [
                c.strip() for c in str(row.get('categories', '')).split(',')
                if c.strip().lower() not in ('food', 'restaurants')
            ]
            rev_count = int(row.get('review_count', 0) or 0)
        except Exception:
            cuisine   = []
            rev_count = 0

        # Cosine distance [0,2] → similarity [0,1]
        similarity_score = max(0.0, min(1.0, 1 - (dist / 2.0)))

        output.append({
            'name':             meta.get('name', 'Unknown'),
            'address':          meta.get('address', ''),
            'city':             meta.get('city', ''),
            'rating':           float(meta.get('rating', 0.0) or 0.0),
            'price':            int(meta.get('price', 0) or 0),   # int, not '$' string
            'cuisine':          cuisine,
            'review_count':     rev_count,
            'similarity_score': similarity_score,
        })

    """
    if verbose:
        print(f"\n🍽️  Top {len(output)} results for: '{query}'\n")
        for i, r in enumerate(output, 1):
            price_str = '$' * r['price'] if r['price'] else 'N/A'
            print(f"{i}. {r['name']} ({r['city']})")
            print(f"   ⭐ {r['rating']}  {price_str}  📍 {r['address']}")
            print(f"   Match: {r['similarity_score']:.1%}\n")
    """        

    return output 


In [7]:
# ─────────────────────────────────────────
# 6. RECOMMEND
# ─────────────────────────────────────────
def recommend_restaurants(query, collection, embedding_model, df, top_k=5):
    print(f"\n{'='*60}")
    print(f"USER QUERY: {query}")
    print(f"{'='*60}")

    recommendations = search_restaurants(
        query, collection, embedding_model, df,
        top_k=top_k, verbose=True
    )

    if not recommendations:
        print("No restaurants found matching your criteria.")
        print("Try relaxing some filters or using different keywords.")
        return []

    print(f"Found {len(recommendations)} restaurant(s):\n")

    for i, r in enumerate(recommendations, 1):
        rating    = float(r.get('rating', 0) or 0)
        price     = int(r.get('price', 0) or 0)
        rev_count = int(r.get('review_count', 0) or 0)
        cuisine   = r.get('cuisine', [])

        print(f"{i}. {r['name']}  —  {r['city']}")
        print(f"   {'⭐' * int(rating)} {rating}/5  ({rev_count} reviews)")
        print(f"   Cuisine : {', '.join(cuisine) if cuisine else 'N/A'}")
        print(f"   Price   : {'$' * price if price else 'N/A'}")
        print(f"   Address : {r['address']}")
        print(f"   Match   : {r['similarity_score']:.1%}")
        print()

    return recommendations

In [14]:
# ─────────────────────────────────────────
# 7. TEST QUERIES
# ─────────────────────────────────────────
TEST_QUERIES = [
    # Price-focused
    {"query": "cheap tacos in santa barbara",       "expected_price": [1, 2], "expected_cuisine": ["tacos"]},
    {"query": "affordable breakfast spot",           "expected_price": [1, 2], "expected_cuisine": ["breakfast"]},
    {"query": "fine dining for a special occasion",  "expected_price": [3, 4], "expected_cuisine": []},
    {"query": "budget sushi",                        "expected_price": [1, 2], "expected_cuisine": ["sushi"]},
    # Cuisine-focused
    {"query": "best ramen in goleta",               "expected_price": None,   "expected_cuisine": ["ramen"]},
    {"query": "vegan friendly cafe",                "expected_price": None,   "expected_cuisine": ["vegan", "cafe"]},
    {"query": "mexican food in carpinteria",        "expected_price": None,   "expected_cuisine": ["mexican"]},
    {"query": "coffee shop to work from",           "expected_price": None,   "expected_cuisine": ["coffee"]},
    # Semantic / ambience
    {"query": "romantic dinner with outdoor seating","expected_price": None,   "expected_cuisine": []},
    {"query": "family friendly brunch spot",        "expected_price": None,   "expected_cuisine": ["brunch"]},
    {"query": "sports bar with happy hour",         "expected_price": None,   "expected_cuisine": ["sports bar"]},
    {"query": "dog friendly patio restaurant",      "expected_price": None,   "expected_cuisine": []},
]

def run_test_queries(collection, embedding_model, df):
    print("\n" + "=" * 60)
    print("RUNNING TEST QUERIES")
    print("=" * 60)
    for i, test in enumerate(TEST_QUERIES, 1):
        print(f"\nTest {i}: {test['query']}")
        results = recommend_restaurants(
            test['query'], collection, embedding_model, df,
            top_k=5)
        print(f"  → {[r['name'] for r in results]}")

In [15]:
# ─────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_csv("nyctastematch-restaurant-recommender/data/processed/CA_restaurants_cleaned.csv")
    print(f"Loaded {len(df)} restaurants")

    # Fix city whitespace
    df["city"] = df["city"].str.replace(r"\s+", " ", regex=True).str.strip()

    # Fill NaN prices with 0 (unknown) so description builder skips them cleanly
    df["price"] = df["price"].fillna(0).astype(int)

    # Generate descriptions + embeddings
    embedding_model, embeddings = generate_embeddings(df)

    # Build ChromaDB collection
    client, collection = recreate_collection_with_cosine(df, embeddings)

    # Run test queries
    run_test_queries(collection, embedding_model, df)

Loaded 1036 restaurants

Sample description:
Restaurant name: Helena Avenue Bakery. Located in Santa Barbara, California. Cuisine types: salad, coffee and tea, breakfast and brunch, sandwiches, bakeries. Price range: Moderate. Rating: 4.0/5 stars. Popular restaurant. Location: 131 Anacapa St, Ste C. Ambience: hipster, trendy, casual. Noise level: average. Good for: lunch, brunch, breakfast. Features: outdoor seating, takeout available, good for kids, dog friendly, wheelchair accessible. Open late


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4351.60it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Generating embeddings...


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 33/33 [00:10<00:00,  3.06it/s]


Embeddings shape: (1036, 384)
Saved → ca_restaurant_embeddings.npy
Recreating ChromaDB collection...
Deleted old collection
✓ Created collection with cosine distance
  Batch 1/3 added
  Batch 2/3 added
  Batch 3/3 added

✅ Collection ready: 1036 restaurants

RUNNING TEST QUERIES

Test 1: cheap tacos in santa barbara

USER QUERY: cheap tacos in santa barbara

PARSED QUERY
💰 Price Range: $, $$
🍽️  Cuisines: tacos
📍 City: Santa Barbara
🔍 Semantic Query: 'cheap tacos in'

Found 5 restaurant(s):

1. Lilly's Tacos  —  Santa Barbara
   ⭐⭐⭐⭐ 4.5/5  (1219 reviews)
   Cuisine : Mexican
   Price   : $
   Address : 310 Chapala St
   Match   : 77.7%

2. Taco Bell  —  Santa Barbara
   ⭐⭐⭐ 3.5/5  (30 reviews)
   Cuisine : Breakfast & Brunch, Fast Food, Tex-Mex, Mexican, Tacos
   Price   : $
   Address : 821 N Milpas Street
   Match   : 76.8%

3. East Beach Tacos  —  Santa Barbara
   ⭐⭐⭐⭐ 4.5/5  (856 reviews)
   Cuisine : Korean, American (New), Mexican, Tacos, Nightlife, Fast Food, Bars
   Price   : 

In [16]:


# ─────────────────────────────────────────
# GROUND TRUTH TEST QUERIES
# A result is "relevant" if it matches the
# expected cuisine AND/OR price tier.
# Based on actual results observed above.
# ─────────────────────────────────────────
TEST_QUERIES = [
    {
        "query": "cheap tacos in santa barbara",
        "relevant_names": ["Lilly's Tacos", "East Beach Tacos", "Mayo's Carniceria & Tacos",
                           "Taco Bell", "La Super-Rica Taqueria", "Oaxaca Fresh",
                           "Taqueria Rincon Alteno", "El Taco Grande"],
        "expected_price": [1, 2],
        "expected_cuisine": ["tacos", "mexican"],
    },
    {
        "query": "affordable breakfast spot",
        "relevant_names": ["Backyard Bowls", "Lucky Penny", "Crushcakes & Cafe",
                           "The Shop Kitchen", "Outpost", "Helena Avenue Bakery",
                           "Cajé", "Chad's"],
        "expected_price": [1, 2],
        "expected_cuisine": ["breakfast", "brunch", "cafe", "bakery"],
    },
    {
        "query": "fine dining for a special occasion",
        "relevant_names": ["Tydes Restaurant", "Stonehouse Restaurant", "The Lark",
                           "Bouchon Santa Barbara", "Ca'Dario", "Finch & Fork",
                           "Lucky's Steakhouse", "Olio e Limone"],
        "expected_price": [3, 4],
        "expected_cuisine": [],
    },
    {
        "query": "budget sushi",
        "relevant_names": ["Shintori Sushi Factory", "Kai Sushi Shabu-Shabu",
                           "Sushi Teri", "Sushi Teri House", "Sushi Ai",
                           "Arigato Sushi", "Sakana"],
        "expected_price": [1, 2],
        "expected_cuisine": ["sushi", "japanese"],
    },
    {
        "query": "best ramen in goleta",
        "relevant_names": ["Nikka Ramen"],
        "expected_price": None,
        "expected_cuisine": ["ramen"],
    },
    {
        "query": "vegan friendly cafe",
        "relevant_names": ["The Natural Cafe", "Oliver's", "Hana Kitchen",
                           "Deja Vu Cafe IV", "Backyard Bowls", "Loquita",
                           "Cajé", "Crushcakes & Cafe"],
        "expected_price": None,
        "expected_cuisine": ["vegan", "vegetarian", "cafe"],
    },
    {
        "query": "mexican food in carpinteria",
        "relevant_names": ["Oaxaca Fresh", "Delgado's Mexican Foods",
                           "El Taco Grande", "Taco Bell", "Taqueria Rincon Alteno"],
        "expected_price": None,
        "expected_cuisine": ["mexican", "tacos"],
    },
    {
        "query": "coffee shop to work from",
        "relevant_names": ["Old Town Coffee Downtown", "Starbucks",
                           "Hustle & Grind Coffee Company",
                           "The Coffee Bean & Tea Leaf", "Cajé",
                           "Deja Vu Cafe IV", "Caffè"],
        "expected_price": None,
        "expected_cuisine": ["coffee", "cafe"],
    },
    {
        "query": "romantic dinner with outdoor seating",
        "relevant_names": ["Sunset Grille", "Courtyard Cafe", "The Garden Market",
                           "Stonehouse Restaurant", "Tydes Restaurant",
                           "Bouchon Santa Barbara", "Loquita"],
        "expected_price": None,
        "expected_cuisine": [],
    },
    {
        "query": "family friendly brunch spot",
        "relevant_names": ["Chad's", "The Roundhouse", "Finch & Fork",
                           "Backyard Bowls", "Outpost", "Helena Avenue Bakery",
                           "Lucky Penny", "Crushcakes & Cafe"],
        "expected_price": None,
        "expected_cuisine": ["brunch", "breakfast"],
    },
    {
        "query": "sports bar with happy hour",
        "relevant_names": ["Dublin's Sports Grill", "The Good Bar",
                           "The Honor Bar", "Uptown Bar & Lounge",
                           "The Press Room", "Wildcat Lounge"],
        "expected_price": None,
        "expected_cuisine": ["sports bar", "bar"],
    },
    {
        "query": "dog friendly patio restaurant",
        "relevant_names": ["Petrini's Italian Restaurant - Santa Barbara",
                           "The Garden Market", "Loquita", "Courtyard Cafe",
                           "Helena Avenue Bakery", "Backyard Bowls",
                           "Feast: The Cafe at Field + Fort"],
        "expected_price": None,
        "expected_cuisine": [],
    },
]


# ─────────────────────────────────────────
# METRIC HELPERS  (same as NYC version)
# ─────────────────────────────────────────
def precision_at_k(retrieved: list, relevant: set, k: int = 5) -> float:
    retrieved_k = retrieved[:k]
    hits = sum(1 for r in retrieved_k if r in relevant)
    return hits / k


def ndcg_at_k(retrieved: list, relevant: set, k: int = 5) -> float:
    dcg = sum(
        1 / np.log2(i + 2)
        for i, r in enumerate(retrieved[:k])
        if r in relevant
    )
    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def hit_rate_at_k(retrieved: list, relevant: set, k: int = 5) -> float:
    return float(any(r in relevant for r in retrieved[:k]))


def reciprocal_rank(retrieved: list, relevant: set) -> float:
    for i, r in enumerate(retrieved):
        if r in relevant:
            return 1 / (i + 1)
    return 0.0


# ─────────────────────────────────────────
# EVALUATION RUNNER
# ─────────────────────────────────────────
def evaluate_rag_model(collection, embedding_model, df, k: int = 5):
    print("\n" + "=" * 70)
    print(f"CA RAG MODEL EVALUATION  (k={k})")
    print("=" * 70)

    all_precision, all_ndcg, all_hit, all_rr = [], [], [], []

    for test in TEST_QUERIES:
        results = search_restaurants(
            test["query"], collection, embedding_model, df,
            top_k=k, verbose=False
        )
        retrieved_names = [r["name"] for r in results]
        relevant_set    = set(test["relevant_names"])

        p   = precision_at_k(retrieved_names, relevant_set, k)
        n   = ndcg_at_k(retrieved_names, relevant_set, k)
        h   = hit_rate_at_k(retrieved_names, relevant_set, k)
        rr  = reciprocal_rank(retrieved_names, relevant_set)

        all_precision.append(p)
        all_ndcg.append(n)
        all_hit.append(h)
        all_rr.append(rr)

        status = "✅" if h else "❌"
        print(f"\n{status} Query: '{test['query']}'")
        print(f"   Retrieved : {retrieved_names}")
        print(f"   Precision@{k}: {p:.2f}  NDCG@{k}: {n:.2f}  "
              f"Hit: {int(h)}  RR: {rr:.2f}")

    print("\n" + "=" * 70)
    print("AGGREGATE RESULTS")
    print("=" * 70)
    print(f"  Precision@{k} : {np.mean(all_precision):.3f}")
    print(f"  NDCG@{k}      : {np.mean(all_ndcg):.3f}")
    print(f"  Hit Rate@{k}  : {np.mean(all_hit):.3f}")
    print(f"  MRR          : {np.mean(all_rr):.3f}")
    print("=" * 70)

    return {
        "precision": np.mean(all_precision),
        "ndcg":      np.mean(all_ndcg),
        "hit_rate":  np.mean(all_hit),
        "mrr":       np.mean(all_rr),
    }


# ─────────────────────────────────────────
# RUN
# ─────────────────────────────────────────
metrics = evaluate_rag_model(collection, embedding_model, df, k=5)


CA RAG MODEL EVALUATION  (k=5)

✅ Query: 'cheap tacos in santa barbara'
   Retrieved : ["Lilly's Tacos", 'Taco Bell', 'East Beach Tacos', "Mayo's Carniceria & Tacos", 'Taco Bell']
   Precision@5: 1.00  NDCG@5: 1.00  Hit: 1  RR: 1.00

✅ Query: 'affordable breakfast spot'
   Retrieved : ['Backyard Bowls', 'Lucky Penny', 'Crushcakes & Cafe', 'The Shop Kitchen', 'Outpost']
   Precision@5: 1.00  NDCG@5: 1.00  Hit: 1  RR: 1.00

✅ Query: 'fine dining for a special occasion'
   Retrieved : ['Feast and Fest Catering', 'Tydes Restaurant', 'Simply Marvelous BBQ Catering', 'Stonehouse Restaurant', 'Market Forays']
   Precision@5: 0.40  NDCG@5: 0.36  Hit: 1  RR: 0.50

✅ Query: 'budget sushi'
   Retrieved : ['Shintori Sushi Factory', 'Kai Sushi Shabu-Shabu', 'Sushi Teri', 'Sushi Teri House', 'Sushi Ai']
   Precision@5: 1.00  NDCG@5: 1.00  Hit: 1  RR: 1.00

✅ Query: 'best ramen in goleta'
   Retrieved : ['Nikka Ramen', "Roundin' Third", 'Draughtsmen Aleworks', 'Top Shelf Event Services', 'Backyard B